In [1]:
import pandas as pd
import numpy as np

complications = pd.read_csv(
    'Complications_and_Deaths-Hospital.csv',
    dtype={'Facility ID': str},
    low_memory=False
)

print(f"Raw shape: {complications.shape}")
complications.head()

Raw shape: (95780, 18)


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Measure ID,Measure Name,Compared to National,Denominator,Score,Lower Estimate,Higher Estimate,Footnote,Start Date,End Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,COMP_HIP_KNEE,Rate of complications for hip/knee replacement...,No Different Than the National Rate,27,3.2,1.7,5.9,NaN,04/01/2021,03/31/2024
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Hybrid_HWM,Hybrid Hospital-Wide All-Cause Risk Standardiz...,No Different Than the National Rate,1835,4.5,2.6,7.4,NaN,07/01/2023,06/30/2024
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,MORT_30_AMI,Death rate for heart attack patients,No Different Than the National Rate,270,11.4,9.1,14.3,NaN,07/01/2021,06/30/2024
3,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,MORT_30_CABG,Death rate for CABG surgery patients,No Different Than the National Rate,144,3,1.6,5.8,NaN,07/01/2021,06/30/2024
4,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,MORT_30_COPD,Death rate for COPD patients,No Different Than the National Rate,112,9.4,6.4,13.6,NaN,07/01/2021,06/30/2024


In [2]:
print("--- Raw nulls ---")
print(complications.isnull().sum())

print("\n--- Raw dtypes ---")
print(complications.dtypes)

--- Raw nulls ---
Facility ID                 0
Facility Name               0
Address                     0
City/Town                   0
State                       0
ZIP Code                    0
County/Parish               0
Telephone Number            0
Measure ID                  0
Measure Name                0
Compared to National        0
Denominator                 0
Score                       0
Lower Estimate              0
Higher Estimate             0
Footnote                50908
Start Date                  0
End Date                    0
dtype: int64

--- Raw dtypes ---
Facility ID             object
Facility Name           object
Address                 object
City/Town               object
State                   object
ZIP Code                 int64
County/Parish           object
Telephone Number        object
Measure ID              object
Measure Name            object
Compared to National    object
Denominator             object
Score                   object
Lower 

In [3]:
complications.columns = (
    complications.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.replace(r'[^\w]', '_', regex=True)
)

print(complications.columns.tolist())

['facility_id', 'facility_name', 'address', 'city_town', 'state', 'zip_code', 'county_parish', 'telephone_number', 'measure_id', 'measure_name', 'compared_to_national', 'denominator', 'score', 'lower_estimate', 'higher_estimate', 'footnote', 'start_date', 'end_date']


In [4]:
CMS_NULLS = {
    'Not Available': np.nan,
    'Not Applicable': np.nan,
    'Too Few to Report': np.nan,
    'N/A': np.nan
}

complications = complications.replace(CMS_NULLS)

print("CMS placeholders replaced ")
print(f"Nulls after replace:\n{complications.isnull().sum()}")

CMS placeholders replaced 
Nulls after replace:
facility_id                 0
facility_name               0
address                     0
city_town                   0
state                       0
zip_code                    0
county_parish               0
telephone_number            0
measure_id                  0
measure_name                0
compared_to_national    31720
denominator             46566
score                   43646
lower_estimate          43646
higher_estimate         43646
footnote                50908
start_date                  0
end_date                    0
dtype: int64


In [5]:
complications['facility_id'] = (
    complications['facility_id']
    .str.strip()
    .str.zfill(6)
)

print("facility_id sample:", complications['facility_id'].head().tolist())

facility_id sample: ['010001', '010001', '010001', '010001', '010001']


In [6]:
numeric_cols = ['score', 'denominator', 'lower_estimate', 'higher_estimate']

for col in numeric_cols:
    complications[col] = pd.to_numeric(complications[col], errors='coerce')

print("Numeric dtypes after cast:")
print(complications[numeric_cols].dtypes)

Numeric dtypes after cast:
score              float64
denominator        float64
lower_estimate     float64
higher_estimate    float64
dtype: object


In [7]:
complications['is_suppressed'] = complications['footnote'].notna()

print("Suppression breakdown:")
print(complications['is_suppressed'].value_counts())

Suppression breakdown:
is_suppressed
False    50908
True     44872
Name: count, dtype: int64


In [8]:
complications['compared_to_national'] = (
    complications['compared_to_national']
    .str.strip()
)

print("compared_to_national values:")
print(complications['compared_to_national'].value_counts(dropna=False))

compared_to_national values:
compared_to_national
No Different Than the National Rate     46793
NaN                                     31720
Number of Cases Too Small               11926
No Different Than the National Value     2697
Better Than the National Rate            1296
Worse Than the National Rate             1125
Worse Than the National Value             162
Better Than the National Value             61
Name: count, dtype: int64


In [9]:
cols_to_drop = [
    'facility_name', 'address', 'city_town',
    'state', 'zip_code', 'county_parish', 'telephone_number',
    'start_date', 'end_date',  
    'footnote'                 
]

complications = complications.drop(columns=cols_to_drop, errors='ignore')

print(f"Shape after dropping: {complications.shape}")
print(complications.columns.tolist())

Shape after dropping: (95780, 9)
['facility_id', 'measure_id', 'measure_name', 'compared_to_national', 'denominator', 'score', 'lower_estimate', 'higher_estimate', 'is_suppressed']


In [10]:
dupes = complications.duplicated(subset=['facility_id', 'measure_id']).sum()
print(f"True duplicates on composite key: {dupes}")

True duplicates on composite key: 0


In [11]:
print(f"Final shape      : {complications.shape}")
print(f"Unique hospitals : {complications['facility_id'].nunique()}")
print(f"Unique measures  : {complications['measure_id'].nunique()}")

print("\n--- Nulls ---")
print(complications.isnull().sum())

print("\n--- Measures available ---")
print(complications['measure_id'].value_counts())

Final shape      : (95780, 9)
Unique hospitals : 4789
Unique measures  : 20

--- Nulls ---
facility_id                 0
measure_id                  0
measure_name                0
compared_to_national    31720
denominator             46566
score                   43646
lower_estimate          43646
higher_estimate         43646
is_suppressed               0
dtype: int64

--- Measures available ---
measure_id
COMP_HIP_KNEE    4789
Hybrid_HWM       4789
PSI_15           4789
PSI_14           4789
PSI_13           4789
PSI_12           4789
PSI_11           4789
PSI_10           4789
PSI_09           4789
PSI_08           4789
PSI_06           4789
PSI_04           4789
PSI_03           4789
MORT_30_STK      4789
MORT_30_PN       4789
MORT_30_HF       4789
MORT_30_COPD     4789
MORT_30_CABG     4789
MORT_30_AMI      4789
PSI_90           4789
Name: count, dtype: int64


In [12]:
comp_scores = complications.pivot_table(
    index='facility_id',
    columns='measure_id',
    values='score',
    aggfunc='first'
).reset_index()

comp_scores.columns = (
    ['facility_id'] +
    ['comp_score_' + col.lower() for col in comp_scores.columns[1:]]
)

# Pivot 2: compared_to_national flag per measure
comp_flags = complications.pivot_table(
    index='facility_id',
    columns='measure_id',
    values='compared_to_national',
    aggfunc='first'
).reset_index()

comp_flags.columns = (
    ['facility_id'] +
    ['comp_flag_' + col.lower() for col in comp_flags.columns[1:]]
)

# Merge both pivots
complications_wide = comp_scores.merge(comp_flags, on='facility_id', how='left')

print(f"Wide table shape: {complications_wide.shape}")
complications_wide.head()

Wide table shape: (4184, 41)


,facility_id,comp_score_comp_hip_knee,comp_score_hybrid_hwm,comp_score_mort_30_ami,comp_score_mort_30_cabg,comp_score_mort_30_copd,comp_score_mort_30_hf,comp_score_mort_30_pn,comp_score_mort_30_stk,comp_score_psi_03,...,comp_flag_psi_06,comp_flag_psi_08,comp_flag_psi_09,comp_flag_psi_10,comp_flag_psi_11,comp_flag_psi_12,comp_flag_psi_13,comp_flag_psi_14,comp_flag_psi_15,comp_flag_psi_90
0,010001,3.2,4.5,11.4,3.0,9.4,10.2,18.4,13.5,0.13,...,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,Worse Than the National Rate,No Different Than the National Value
1,010005,3.0,4.6,NaN,NaN,8.9,14.1,21.2,12.9,0.33,...,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Value
2,010006,4.7,5.2,14.5,5.4,8.7,12.5,19.6,12.4,0.55,...,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,Worse Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Value
3,010007,NaN,4.8,NaN,NaN,11.2,13.4,25.4,NaN,0.54,...,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,Number of Cases Too Small,Number of Cases Too Small,No Different Than the National Rate,No Different Than the National Value
4,010008,NaN,4.3,NaN,NaN,NaN,NaN,15.0,NaN,0.60,...,No Different Than the National Rate,No Different Than the National Rate,NaN,NaN,NaN,Number of Cases Too Small,NaN,NaN,NaN,NaN


In [13]:
complications_wide.to_csv('complications_cleaned.csv', index=False)
print("Saved as complications_cleaned.csv")

Saved as complications_cleaned.csv


In [14]:
print(complications_wide.columns.tolist())

['facility_id', 'comp_score_comp_hip_knee', 'comp_score_hybrid_hwm', 'comp_score_mort_30_ami', 'comp_score_mort_30_cabg', 'comp_score_mort_30_copd', 'comp_score_mort_30_hf', 'comp_score_mort_30_pn', 'comp_score_mort_30_stk', 'comp_score_psi_03', 'comp_score_psi_04', 'comp_score_psi_06', 'comp_score_psi_08', 'comp_score_psi_09', 'comp_score_psi_10', 'comp_score_psi_11', 'comp_score_psi_12', 'comp_score_psi_13', 'comp_score_psi_14', 'comp_score_psi_15', 'comp_score_psi_90', 'comp_flag_comp_hip_knee', 'comp_flag_hybrid_hwm', 'comp_flag_mort_30_ami', 'comp_flag_mort_30_cabg', 'comp_flag_mort_30_copd', 'comp_flag_mort_30_hf', 'comp_flag_mort_30_pn', 'comp_flag_mort_30_stk', 'comp_flag_psi_03', 'comp_flag_psi_04', 'comp_flag_psi_06', 'comp_flag_psi_08', 'comp_flag_psi_09', 'comp_flag_psi_10', 'comp_flag_psi_11', 'comp_flag_psi_12', 'comp_flag_psi_13', 'comp_flag_psi_14', 'comp_flag_psi_15', 'comp_flag_psi_90']
